# <center> 212. Word Search II </center>


## Problem Description
[Click here](https://leetcode.com/problems/word-search-ii/description/)


## Intuition
<!-- Describe your first thoughts on how to solve this problem. -->
Similar to [Word Search I](https://leetcode.com/problems/word-search/description/), the difference is that instead of searching for a single word, we must find all words from a given list. 

In word seach I, we use DFS + backtracking to search for a word on the board. A brute-force approach is to run the Word Search I algorithm for every word, resulting in O(x × m × n × 3^w) time, where x is the number of words and w is the maximum word length.

To avoid repeatedly exploring the same prefixes, build a Trie containing all words. During DFS (backtracking), only continue along paths that exist in the Trie, which prunes invalid searches early.

Treat each cell as a starting point. If a word is found, add it to the result set and remove it from the Trie.

To avoid revisiting the same cell, mark it as visited before exploring its neighbors and unmark it afterward. For marking, either use a hash set or temporarily replace the cell letter with a special character (you can use any character except an English letter).


## Approach
<!-- Describe your approach to solving the problem. -->
**Class TrieNode**

**init()**
- set children = a hashmap to store the child nodes
    - *key = characters* 
    - *value = child nodes*
- set end = none to track if the node is the last node (character) of a word. If the node is the last node, the end attribute will contain the word

**Class Solution**

**findWords()**
- set m = no. of rows and n = no. of columns
- set res = a set to store the result
- store the words in a trie i.e for each letter in a word, create a trie node
    - create a root node
    - for each word
        - set cur = root to track current node
        - for each character in the word
            - if the character is not a child node of the current node, add it as the child node
            - else set the current node to the child node of the character to move to the next node
        - at the end of the loop, the current node will point to the last node (character) of the word. So, store the current word in the end attribute
- define a helper function for backtracking <br>
*dfs(row, column, current node)*
    - if you found all words or the current node is null, return
    - if current node is not the last node and the node has no child nodes, set current node to null and return
    - if the row or col is out of bounds or the cell has been visited, return
    - get the current cell letter
    - if letter is not in the child nodes of current node, return
    - else get the child node
    - if the child node is the last node
        - add the word to result
        - set end attribute as none to remove the word from trie and search space
    - mark the current as visited
    - Search the 4 neighbor cells (top, bottom, right, left)
    - unmark the cell 
- for each cell, do backtracking
- return result set


## Complexity
<!-- Add your time complexity here, e.g. $$O(n)$$ -->
- Time complexity: 
  - init(): O(1)
  - findWords(): O(building trie + board traversal * backtracking) → O(total words * max-length word + rows * cols * total recursive calls * work per call) → O(total words * max-length word + rows * cols * total recursion tree nodes * constant) → O(x * w + m * n * 4 * $3^{w-1}$ * constant) → O(x * w + m * n * $3^w$) → O(m * n * $3^w$)
    - *we can start the search from every cell, giving m × n possible starting positions*
    - *branching factor varies but follows the pattern (4, 3, 3, ...), so recursion tree contains at most 4 × $3^{w-1}$ nodes because:*
      - *from the starting cell, we can explore up to 4 directions*
      - *after the first move, we have at most 3 choices because we can't revisit the previous cell*
    - *as branching factor varies with a fixed pattern, total recursive calls = total recursion tree nodes*

<!-- Add your space complexity here, e.g. $$O(n)$$ -->
- Space complexity: 
  - init(): O(1)
  - findWords(): O(trie + result set + recursion stack) → O(trie + result set + recursion depth) → O(total words * max-length word + total words + recursion depth) → O(x * w + x + w) → O(x * w)
    - *trie requires O(x × w) space because a node may be created for each character*
    - *recursion depth is w because we process one character per level*

## Code

In [ ]:
class TrieNode:

    def __init__(self):
        self.children = defaultdict(TrieNode)
        self.end = None


class Solution:

    def findWords(self, board: List[List[str]], words: List[str]) -> List[str]:
        m, n = len(board), len(board[0])
        res = set()

        root = TrieNode()
        for word in words:
            cur = root
            for char in word:
                if char not in cur.children:
                    cur.children[char] = TrieNode()
                cur = cur.children[char]
            cur.end = word

        def backtrack(row: int, col: int, node: TrieNode) -> None:
            if len(res) == len(words) or not node:   
                return
            if not node.end and not node.children:  
                node = None  
                return
            if row < 0 or row == m or col < 0 or col == n or board[row][col] == '*':
                return
            c = board[row][col]
            if c not in node.children:
                return
            child = node.children[c]
            if child.end:
                res.add(child.end)
                child.end = None
            board[row][col] = '*'
            backtrack(row + 1, col, child)
            backtrack(row - 1, col, child)
            backtrack(row, col + 1, child)
            backtrack(row, col - 1, child)
            board[row][col] = c

        for row in range(m):
            for col in range(n):
                backtrack(row, col, root)
        return res